## Test install

In [8]:
import sys 
print(sys.executable) 
print(sys.version)

/home/dali/WORK/Challenges/relais/.venv/bin/python
3.13.7 (main, Mar  3 2026, 12:19:54) [GCC 15.2.0]


# morse decoder

In [ ]:
import numpy as np
from scipy.io import wavfile

MORSE = {
    ".-": "A", "-...": "B", "-.-.": "C", "-..": "D", ".": "E",
    "..-.": "F", "--.": "G", "....": "H", "..": "I", ".---": "J",
    "-.-": "K", ".-..": "L", "--": "M", "-.": "N", "---": "O",
    ".--.": "P", "--.-": "Q", ".-.": "R", "...": "S", "-": "T",
    "..-": "U", "...-": "V", ".--": "W", "-..-": "X", "-.--": "Y",
    "--..": "Z",
    "-----": "0", ".----": "1", "..---": "2", "...--": "3",
    "....-": "4", ".....": "5", "-....": "6", "--...": "7",
    "---..": "8", "----.": "9",
}


def load_wav(filename):
    sample_rate, audio = wavfile.read(filename)

    # Stereo -> mono
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    audio = audio.astype(np.float64)

    # Normalize
    max_value = np.max(np.abs(audio))
    if max_value > 0:
        audio /= max_value

    return sample_rate, audio


def detect_tone(audio, sample_rate, threshold=0.15):
    """
    Detect whether each sample contains the Morse tone.
    """
    return np.abs(audio) > threshold


def run_lengths(signal):
    """
    Return (state, number_of_samples) for consecutive states.
    """
    changes = np.diff(signal.astype(np.int8))
    positions = np.where(changes != 0)[0] + 1

    starts = np.r_[0, positions]
    ends = np.r_[positions, len(signal)]

    return [
        (signal[start], end - start)
        for start, end in zip(starts, ends)
    ]


def decode_morse_wav(filename):
    sample_rate, audio = load_wav(filename)

    tone = detect_tone(audio, sample_rate)

    runs = run_lengths(tone)

    # Remove tiny noise pulses
    min_samples = int(sample_rate * 0.005)

    runs = [
        (state, length)
        for state, length in runs
        if length >= min_samples
    ]

    # Find tone durations
    tone_lengths = [
        length / sample_rate
        for state, length in runs
        if state
    ]

    if not tone_lengths:
        return ""

    # Estimate basic Morse unit.
    # Shortest tone is approximately 1 unit.
    unit = np.percentile(tone_lengths, 20)

    result = []
    current_letter = []

    for state, length_samples in runs:
        duration = length_samples / sample_rate

        if state:
            # Tone
            if duration < unit * 2:
                current_letter.append(".")
            else:
                current_letter.append("-")

        else:
            # Silence
            if duration < unit * 2:
                # Between dots/dashes in same letter
                continue

            elif duration < unit * 5:
                # Between letters
                if current_letter:
                    code = "".join(current_letter)
                    result.append(MORSE.get(code, "?"))
                    current_letter = []

            else:
                # Between words
                if current_letter:
                    code = "".join(current_letter)
                    result.append(MORSE.get(code, "?"))
                    current_letter = []

                result.append(" ")

    if current_letter:
        code = "".join(current_letter)
        result.append(MORSE.get(code, "?"))

    return "".join(result)

# Test

In [12]:
print("Start")
text = decode_morse_wav("signal-01.wav")
print(text)

Start



In [ ]:
from scipy.io import wavfile
import numpy as np

filename = "signal-01.wav"

sample_rate, audio = wavfile.read(filename)

print("Sample rate:", sample_rate)
print("Shape:", audio.shape)
print("Dtype:", audio.dtype)
print("Duration:", len(audio) / sample_rate, "seconds")

if audio.ndim > 1:
    audio = audio.mean(axis=1)

print("Min:", audio.min())
print("Max:", audio.max())
print("RMS:", np.sqrt(np.mean(audio.astype(float) ** 2)))


KeyboardInterrupt

